#### Fabric Demo — Energy & OEE Simulator
##### Notebook | Connects to SQL Server via Managed Private Endpoint

In [3]:
# Simulator settings
interval_seconds   = 5       # How often to insert a batch of rows
machines_per_batch = 10      # One row per machine per tick

# Event timing (in loop iterations)
spike_start_iter   = 30      # ~2.5 min at 5s intervals
spike_end_iter     = 84      # ~7 min duration
degrade_start_iter = 48      # starts at ~4 min # Degradation is open-ended — runs until notebook is stopped

variables = notebookutils.variableLibrary.getLibrary("var_library_rti")

# Machine definitions: id, line, base_kw, base_oee
df_machines = spark.read.format("delta").load(f"{variables.lakehouse_abfss}/dbo/dim_machine")

# Energy tariff (EUR/kWh) — kept simple for demo
tariff_eur_per_kwh = 0.85

# Production units per kW per interval (rough throughput proxy)
units_per_kw_per_tick = 0.05

print("Configuration loaded.")
print(f"Machines: {df_machines.count()}")
print(f"Interval: {interval_seconds}s")
print(f"Spike starts: iteration {spike_start_iter} (~{spike_start_iter * interval_seconds // 60} min)")
print(f"Degrade starts: iteration {degrade_start_iter} (~{degrade_start_iter * interval_seconds // 60} min)")


StatementMeta(, a9aefbd2-5e26-46ea-bfa1-9418c693fe4f, 14, Finished, Available, Finished, False)

Configuration loaded.
Machines: 10
Interval: 5s
Spike starts: iteration 30 (~2 min)
Degrade starts: iteration 48 (~4 min)


In [2]:
# =============================================================================
# SQL Server connection via SQL Authentication
# =============================================================================
import pyodbc
from notebookutils import mssparkutils

# SQL Server connection — update to your managed private endpoint details
sql_server = "demo-sql-fabric.database.windows.net"
sql_database = "sql-fabric-rti"
sql_table = "dbo.EnergyReadings"
username = mssparkutils.credentials.getSecret(
    "https://ms-demo-keyvault.vault.azure.net/",
    "sql-database-username"
    )
password = mssparkutils.credentials.getSecret(
    "https://ms-demo-keyvault.vault.azure.net/",
    "sql-database-password"
    ) 

def get_sql_connection():
    """
    Connect to SQL Server using SQL Authentication.
    """    
    
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={sql_server};"
        f"DATABASE={sql_database};"
        f"UID={username};"
        f"PWD={password};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
        )

    conn = pyodbc.connect(conn_str)
    return conn

# Test connection
try:
    conn = get_sql_connection()
    print("Connected to SQL Server via SQL Authentication.")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")
    raise

StatementMeta(, a9aefbd2-5e26-46ea-bfa1-9418c693fe4f, 13, Finished, Available, Finished, False)

Connected to SQL Server via SQL Authentication.


In [4]:
# =============================================================================
# Simulator loop
# =============================================================================

import time
import random
import math
from datetime import datetime, timezone
from pyspark.sql import functions as F
variables = notebookutils.variableLibrary.getLibrary("var_library_rti")

machines_df = spark.read \
                .format("delta") \
                .load(f"{variables.lakehouse_abfss}/dbo/dim_machine") \
                .select("MachineId", "LineId", "BaseKW", "BaseOEE")

# Collect to driver (dimension table expected to be small)
machines = machines_df.collect()

# -------------------------------------------------------------------------
# Insert template
# -------------------------------------------------------------------------
INSERT_SQL = f"""
INSERT INTO {sql_table} (
    Timestamp, MachineId, PlantId, LineId, ShiftId,
    PowerKw, EnergyKwhCumul, OeePercent, ProductionUnits,
    CostPerKwh, CostThisTick, CostPerUnit, EventTag
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

# -------------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------------
def get_shift(ts: datetime) -> str:
    h = ts.hour
    if 6 <= h < 14:  return "Morning"
    if 14 <= h < 22: return "Afternoon"
    return "Night"

def jitter(value: float, pct: float) -> float:
    """Apply Gaussian noise of ±pct% to value."""
    return max(0.0, value * (1 + random.gauss(0, pct / 100)))

def sigmoid_ramp(iteration: int, start: int, duration: int) -> float:
    """Smooth 0→1 ramp using a sigmoid so spikes look organic, not stepped."""
    x = (iteration - start) / max(duration * 0.3, 1)
    return 1 / (1 + math.exp(-x))

# -------------------------------------------------------------------------
# Cumulative energy + units accumulators per machine (corrected)
# -------------------------------------------------------------------------
cumul_kwh   = {m.MachineId: 0.0 for m in machines}
cumul_units = {m.MachineId: 0   for m in machines}

print("Simulator running. Ctrl+C or stop the notebook cell to halt.")
print(f"{'Iter':>5}  {'Time':>8}  {'Event':<20}  {'Rows':>4}")
print("-" * 36)

iteration = 0
try:
    while True:
        ts_now    = datetime.now(timezone.utc)
        shift_id  = get_shift(ts_now)
        rows      = []

        # Re-read Variable Library each tick to pick up live changes (Phase 5)
        variables = notebookutils.variableLibrary.getLibrary("var_library_rti")
        spike_machine    = getattr(variables, 'spike_machine_id', '').strip()
        spike_multiplier = float(getattr(variables, 'spike_power_multiplier', '1.0'))

        # Determine active events
        spike_active   = spike_start_iter <= iteration < spike_end_iter
        degrade_active = iteration >= degrade_start_iter
        variable_spike_active = bool(spike_machine) and spike_multiplier != 1.0

        # Spike ramp factor (smooth sigmoid, peaks at 2.3×)
        spike_factor = 1.0
        if spike_active:
            ramp = sigmoid_ramp(iteration, spike_start_iter, 12)
            spike_factor = 1.0 + ramp * 1.3   # up to 2.3×

        # OEE degradation factor (linear drift from 1.0 → 0.67 over 3 min)
        oee_degrade_factor = 1.0
        if degrade_active:
            elapsed = min(iteration - degrade_start_iter, 36)   # cap at 3 min
            oee_degrade_factor = 1.0 - (elapsed / 36) * 0.33    # 87% → 58%

        event_label = []
        if variable_spike_active: event_label.append(f"variable_spike({spike_machine})")
        if spike_active:   event_label.append("energy_spike")
        if degrade_active: event_label.append("oee_degrade")
        event_tag = ",".join(event_label) if event_label else None

        # -----------------------------------------------------------------
        # Iterate machines from dbo.dim_machine (corrected)
        # -----------------------------------------------------------------
        for m in machines:
            MachineId = m.MachineId
            LineId    = m.LineId
            BaseKW    = float(m.BaseKW)   if m.BaseKW  is not None else 0.0
            BaseOEE   = float(m.BaseOEE)  if m.BaseOEE is not None else 0.0

            # Apply baseline jitter
            power_kw = jitter(BaseKW,  5)
            oee      = jitter(BaseOEE, 2)
            
            # Apply events to specific machines
            # Variable-driven spike (Phase 5) takes priority over scripted events
            if spike_machine and MachineId == spike_machine and spike_multiplier != 1.0:
                power_kw *= spike_multiplier
                m_tag = "variable_spike"
            elif MachineId == "WELD-L2-A" and spike_active:
                power_kw *= spike_factor
                m_tag = "energy_spike"
            elif MachineId == "COAT-L3-A" and degrade_active:
                oee *= oee_degrade_factor
                m_tag = "oee_degrade"
            else:
                m_tag = None

            # Derived values
            interval_hrs    = interval_seconds / 3600
            kwh_this_tick   = power_kw * interval_hrs
            units_this_tick = max(1, int(BaseKW * units_per_kw_per_tick * (oee / 100)))
            cost_this_tick  = kwh_this_tick * tariff_eur_per_kwh
            cost_per_unit   = cost_this_tick / units_this_tick

            cumul_kwh[MachineId]   += kwh_this_tick
            cumul_units[MachineId] += units_this_tick

            rows.append((
                ts_now,
                MachineId,
                "PLANT-GR-01",
                LineId,
                shift_id,
                round(power_kw, 2),
                round(cumul_kwh[MachineId], 4),
                round(min(oee, 100.0), 2),
                cumul_units[MachineId],
                tariff_eur_per_kwh,
                round(cost_this_tick, 5),
                round(cost_per_unit, 6),
                m_tag,
            ))

        # Batch insert
        conn = get_sql_connection()
        cursor = conn.cursor()
        cursor.executemany(INSERT_SQL, rows)
        conn.commit()
        conn.close()

        # Console heartbeat
        active_str = " + ".join(event_label) if event_label else "baseline"
        print(f"{iteration:>5}  {ts_now.strftime('%H:%M:%S')}  {active_str:<20}  {len(rows):>4}")

        iteration += 1
        time.sleep(interval_seconds)

except KeyboardInterrupt:
    print("\nSimulator stopped.")

StatementMeta(, a9aefbd2-5e26-46ea-bfa1-9418c693fe4f, 15, Finished, Cancelled, Cancelled, True)

Simulator running. Ctrl+C or stop the notebook cell to halt.
 Iter      Time  Event                 Rows
------------------------------------
    0  11:40:26  baseline                10
    1  11:40:34  baseline                10
    2  11:40:42  baseline                10
    3  11:40:50  baseline                10
    4  11:40:57  baseline                10
    5  11:41:05  baseline                10
    6  11:41:13  baseline                10
    7  11:41:21  baseline                10
    8  11:41:29  baseline                10
    9  11:41:37  baseline                10
   10  11:41:45  baseline                10
   11  11:41:53  baseline                10
   12  11:42:01  baseline                10
   13  11:42:08  baseline                10
   14  11:42:17  baseline                10
   15  11:42:24  baseline                10
   16  11:42:32  baseline                10
   17  11:42:40  baseline                10
   18  11:42:48  baseline                10
   19  11:42:56  basel